In [1]:
!pip install tiktoken transformers groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 1.4 MB/s eta 0:00:00


In [2]:
import tiktoken

# Load the tokenizer GPT-4 uses
enc = tiktoken.get_encoding("cl100k_base")

# Tokenize some text
text = "Amazon EC2 provides resizable compute capacity in the cloud."
tokens = enc.encode(text)

print(f"Text: {text}")
print(f"Token count: {len(tokens)}")
print(f"Tokens: {tokens}")
print(f"Decoded back: {enc.decode(tokens)}")

Text: Amazon EC2 provides resizable compute capacity in the cloud.
Token count: 11
Tokens: [26948, 21283, 17, 5825, 98183, 12849, 8824, 304, 279, 9624, 13]
Decoded back: Amazon EC2 provides resizable compute capacity in the cloud.


In [3]:
examples = [
    "Hello world",
    "Kubernetes cluster autoscaling",
    "నమస్కారం",          # Telugu — see how non-English tokenizes
    "AWS::EC2::Instance",  # CloudFormation syntax
    "error: connection refused port 8080",
]

for text in examples:
    tokens = enc.encode(text)
    print(f"'{text}' → {len(tokens)} tokens")

'Hello world' → 2 tokens
'Kubernetes cluster autoscaling' → 5 tokens
'నమస్కారం' → 16 tokens
'AWS::EC2::Instance' → 6 tokens
'error: connection refused port 8080' → 8 tokens


In [4]:
# Groq pricing: roughly $0.05 per 1M tokens (Llama 3.1 8B)
def estimate_cost(text, price_per_million=0.05):
    enc = tiktoken.get_encoding("cl100k_base")
    token_count = len(enc.encode(text))
    cost = (token_count / 1_000_000) * price_per_million
    return token_count, cost

# Test with a typical AWS doc paragraph
aws_doc = """
Amazon Elastic Compute Cloud (Amazon EC2) provides scalable computing
capacity in the Amazon Web Services (AWS) Cloud. Using Amazon EC2
eliminates your need to invest in hardware up front, so you can develop
and deploy applications faster. You can use Amazon EC2 to launch as many
or as few virtual servers as you need, configure security and networking,
and manage storage.
"""

tokens, cost = estimate_cost(aws_doc)
print(f"Token count: {tokens}")
print(f"Cost to process: ${cost:.6f}")
print(f"Cost for 1000 similar paragraphs: ${cost*1000:.4f}")

Token count: 82
Cost to process: $0.000004
Cost for 1000 similar paragraphs: $0.0041


In [5]:
# This is the most important practical lesson
GROQ_LLAMA_CONTEXT = 128_000  # tokens
TYPICAL_CHUNK_SIZE = 512       # tokens
SYSTEM_PROMPT_SIZE = 200       # tokens
ANSWER_RESERVE = 500           # tokens

available_for_chunks = GROQ_LLAMA_CONTEXT - SYSTEM_PROMPT_SIZE - ANSWER_RESERVE
max_chunks_per_request = available_for_chunks // TYPICAL_CHUNK_SIZE

print(f"Available context: {available_for_chunks} tokens")
print(f"Max chunks you can send per request: {max_chunks_per_request}")
print(f"\nThis is why RAG retrieves top 5-10 chunks, not 1000")

Available context: 127300 tokens
Max chunks you can send per request: 248

This is why RAG retrieves top 5-10 chunks, not 1000
